Distribution for two four sided dice:

|2 | 3 | 4 | 5 | 6 | 7 | 8 |
|---|---|---|---|---|---|---|
|1/16|2/16|3/16|4/16|3/16|2/16|1/16|

In [59]:
dice_sides = 4
cut_off = dice_sides+1
high_point = 2*dice_sides +1

probs = 1 / dice_sides

def standard_dist(moves):
    global dice_sides
    global cut_off
    global high_point
    if moves < cut_off: data_val = (moves-1) / dice_sides**2
    else: data_val = (high_point - moves) / dice_sides**2
    return data_val

In [ ]:
import numpy as np
from numpy.linalg import eig
from scipy import sparse

size = 40 #Modelling Jail and Just Visiting as the same - get out of Jail immediately (as though Just Visiting)
rows = []
cols = []
data = []

#30 = Go To Jail
#2, 17, 33 = Community
#7, 22, 36 = chance
special_indices = [30, 2, 7, 17, 22, 33, 36]

In [61]:
#fetching transition matrix probabilities
# IMPORTANT: We are constructing a Transposed Matrix, by switching the columns and rows - purely because (arr.T).dot(t) is faster at multiplying than t @ arr

#if we land on Chance 10): Move 3 steps back; this could potentially lead to positions 4, 18, 33.
# 33 is special index (community), which can cause Go or Jail
#Therefore we  need to do recursion when new_pos==36, new_pos-3 = 33

for i in range(size):
    for moves in range(2,high_point):

        #finding new position on board, accounting for looping around 0
        new_pos = i+moves
        if new_pos>39: new_pos=new_pos%(size-1)

        #calculating standard distribution probability
        data_val = standard_dist(moves)

        #ordinary movement, no special_indices
        if new_pos not in special_indices:
            cols.append(i)
            rows.append(new_pos)
            data.append(data_val) 

        # Now checking for special_indices.
        else:

            #Go To Jail
            if new_pos==30:
                cols.append(i)
                rows.append(10)
                data.append(data_val)

            #chance
            elif new_pos in [7, 22, 36]:

                # 10) Moving 3 steps back, P = 1/16
                n = new_pos - 3
                if n <0: n+=size
                ########checking for recursion here: #####################################
                if n==33:
                    #recursion true. We go to community. Do all community stuff here
                    cols.append(i)
                    rows.append(n)
                    data.append((1/16)*data_val * (14/16))  #14/16 chance of remaining in current position

                    cols.append(i)
                    rows.append(0)
                    data.append((1/16)*data_val * (1/16) ) # 1) 1/16 chance of going to Go [0]

                    cols.append(i)
                    rows.append(10)
                    data.append((1/16)*data_val * (1/16)) # 6)  1/16 chance of going to Jail [10]
                else:
                    cols.append(i)
                    rows.append(n)
                    data.append(data_val * (1/16))
                ###########################################################################


                #6/16 chance of remaining in current position
                cols.append(i)
                rows.append(new_pos)
                data.append( data_val * (6/16))

                # 1)  1/16 for going to Go [0]
                cols.append(i)
                rows.append(0)
                data.append( data_val * (1/16))

                # 2) 1/16 going to Trafalgar square [24]
                cols.append(i)
                rows.append(24)
                data.append(data_val * (1/16))

                # 3) 1/16 going to Mayfair [39]
                cols.append(i)
                rows.append(39)
                data.append(data_val * (1/16))      

                # 4) 1/16 to Pall Mall [11]
                cols.append(i)
                rows.append(11)
                data.append(data_val * (1/16))     

                # 5) and 6) are going to nearest Station. Therefore P = 2/16. There are Stations at positions [5, 15, 25, 35]
                cols.append(i)
                if new_pos%10 >5: rail_loc = new_pos + 15 - (new_pos%10)
                else: rail_loc = new_pos + (5 - (new_pos%10))
                if rail_loc >= size: rail_loc-=size
                rows.append(rail_loc)
                data.append (data_val * (2/16))

                # 7) advance to nearest utility. P = 1/16. Utilities at [12, 28]
                cols.append(i)
                data.append( data_val * (1/16))
                if new_pos==22: rows.append(28)
                else: rows.append(12) #when at 7 or 36, will go to 12

                # 11) 1/16 for going to Jail [10]
                cols.append(i)
                rows.append(10)
                data.append(data_val * (1/16))

                #14) 1/16 for going to Railway Station at [5]
                cols.append(i)
                rows.append(5)
                data.append(data_val * (1/16))
        
            
            #Community
            elif new_pos in [2, 17, 33]:
                cols.append(i)
                rows.append(new_pos)
                data.append(data_val * (14/16))  #14/16 chance of remaining in current position

                cols.append(i)
                rows.append(0)
                data.append(data_val * (1/16) ) # 1) 1/16 chance of going to Go [0]

                cols.append(i)
                rows.append(10)
                data.append(data_val * (1/16)) # 6)  1/16 chance of going to Jail [10]

In [62]:
arr = sparse.csr_matrix((data, (rows, cols)), shape=(size, size))

#checking our Column-Stochastic matrix has columns which actually sum to one
col_sums = np.array(arr.sum(axis=0)).flatten()
print(np.min(col_sums), np.max(col_sums))


#picking matrix iteration, or eigendecomposition methods for calculating stationary dist:
method = "eigen"


######### MATRIX ITERATION METHOD ####################################################################################
# intial marginal distribution t. Can be anything, but I just chose a vector with equal distribution to start
# marginal distribution t should converge to stationary distribution t, over time.
if method == "iteration":
    s = np.ones(size) / size
    for i in range(300):
        new_s = arr.dot(s)

        # convergence detection
        if np.linalg.norm(new_s -s, 1) <= 1e-10:
            print(f"Broken at {i}")
            break
        s = new_s

######## EIGENDECOMPOSITION METHOD #####################################################################################
'''Find eigendecompositions: Whenever eigenvalue==1, we go from (Matrix * vector == eigenvalue * vector) to (Matrix * vector == vector)
For MCs, we have s*P = s, with row stochasticity
And, we have P*s = s, with column stochasticity
We constructed our matrix with column stochasticity, so our life is made easier because we only need to find right eigenvector = stationary_dist (t)'''

if method =="eigen":
    eigenvalues, eigenvectors = np.linalg.eig(arr.toarray())

    #finding the index of the eigenvalue = 1, to get corresponding eigenvector = stationary distribution
    idx = np.argmin(np.abs(eigenvalues-1.0))
    s = (eigenvectors[:, idx]).real

    #normalising stationary distribution
    s = s / s.sum()
#########################################################################################################################


print(np.sum(s))  # should be ~1

1.0 1.0
1.0


In [65]:
# Showing top monopoly locations:
board_london = ["GO","Old Kent Road","Community Chest","Whitechapel Road","Income Tax","King's Cross Station","The Angel Islington","Chance","Euston Road","Pentonville Road","Jail (Just Visiting)","Pall Mall","Electric Company","Whitehall","Northumberland Avenue","Marylebone Station","Bow Street","Community Chest","Marlborough Street","Vine Street","Free Parking","Strand","Chance","Fleet Street","Trafalgar Square","Fenchurch Street Station","Leicester Square","Coventry Street","Water Works","Piccadilly","Go To Jail","Regent Street","Oxford Street","Community Chest","Bond Street","Liverpool Street Station","Chance","Park Lane","Super Tax","Mayfair","Jail 1","Jail 2","Jail 3"]
board_ny = ["GO","Mediterranean Avenue","Community Chest","Baltic Avenue","Income Tax","Reading Railroad","Oriental Avenue","Chance","Vermont Avenue","Connecticut Avenue","Jail (Just Visiting)","St. Charles Place","Electric Company","States Avenue","Virginia Avenue","Pennsylvania Railroad","St. James Place","Community Chest","Tennessee Avenue","New York Avenue","Free Parking","Kentucky Avenue","Chance","Indiana Avenue","Illinois Avenue","B&O Railroad","Atlantic Avenue","Ventnor Avenue","Water Works","Marvin Gardens","Go To Jail","Pacific Avenue","North Carolina Avenue","Community Chest","Pennsylvania Avenue","Short Line Railroad","Chance","Park Place","Luxury Tax","Boardwalk","Jail 1","Jail 2","Jail 3"]
board_india = ["GO", "Guwahati", "Community Chest", "Bhubaneshwar", "Income Tax", "Chennai Central Railway Station", "Panaji (Goa)", "Chance", "Agra", "Vadodara", "Jail (Just Visiting)", "Ludhiana", "Electric Company", "Patna", "Bhopal", "Howrah Railway Station", "Indore", "Community Chest", "Nagpur", "Kochi", "Free Parking", "Lucknow", "Chance", "Chandigarh", "Jaipur", "New Delhi Railway Station", "Pune", "Hyderabad", "Water Company", "Ahmedabad", "Go To Jail", "Kolkata", "Chennai", "Community Chest", "Bengalaru", "Chatrapati Shivaji Terminus", "Chance", "Delhi", "Super Tax", "Mumbai", "Jail 1", "Jail 2", "Jail 3"]

top_matrix_indices = np.argsort(s)[-3:][::-1]

print("The most visited locations\n")
for i in top_matrix_indices: print(i)

The most visited locations

10
15
24
